In [ ]:
from unittest import skip

import pandas as pd
import numpy as np

# Daten einlesen

In [ ]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")

## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [ ]:
df_with_positions = df.copy()
neue_spalten = {}

for col in parteien:
    scores = []

    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']
        if col.startswith('p-') or col.endswith('-pos'):

        # Keine klare Empfehlung oder keine Daten
        if position not in [1.0, 2.0] or pd.isna(ja_proz):
            scores.append(np.nan)

        # Partei empfiehlt JA → je mehr Ja-Stimmen, desto positiver
        elif position == 1.0:
            scores.append((ja_proz - 50) / 100)

        # Partei empfiehlt NEIN → je mehr Nein-Stimmen, desto positiver
        elif position == 2.0:
            scores.append((50 - ja_proz) / 100)

    neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)
print(df_with_positions['zustimmung_br-pos'].dropna().unique())

In [ ]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)

### Datenset für Heatmap

In [ ]:
kanton_cols = [c for c in df.columns if c.endswith('-japroz')]

scores = {}
zeile = df.iloc[70]
for k in kanton_cols:
    ja_proz = pd.to_numeric(zeile[k], errors="coerce")

    if position not in (1.0, 2.0) or pd.isna(ja_proz):
        scores[k] = np.nan
    elif position == 1.0:
        scores[k] = (ja_proz - 50) / 100
    else:  # position == 2.0
        scores[k] = (50 - ja_proz) / 100

scores

# Eine Zeile, Spalten = Kantone (oder Kurzlabels)
br_kantone = pd.DataFrame([scores], index=["br-pos"])
# Optional Spalten umbenennen: zh-japroz -> ZH
br_kantone.columns = [c.replace("-japroz", "").upper() for c in br_kantone.columns]
br_kantone

In [ ]:
kanton_cols = sorted(
    c for c in df.columns if str(c).endswith("-japroz")
)
akteure = ["br-pos"] + sorted(
    c for c in df.columns if str(c).startswith("p-")
)
kanton_labels = [c.replace("-japroz", "").upper() for c in kanton_cols]


def alignment_score(position, ja_proz):
    pos = pd.to_numeric(position, errors="coerce")
    ja = pd.to_numeric(ja_proz, errors="coerce")
    if pos not in (1.0, 2.0) or pd.isna(ja):
        return np.nan
    if pos == 1.0:
        return (ja - 50) / 100
    return (50 - ja) / 100


# --- eine Abstimmung wählen (Beispiele) ---
# zeile = df.iloc[0]
# zeile = df.loc[df["anr"] == 642.0].iloc[0]

zeile = df.loc[df["anr"] == 642.0].iloc[0]  # anpassen

rows = {}
for actor in akteure:
    pos = zeile[actor]
    rows[actor] = {
        lab: alignment_score(pos, zeile[kcol])
        for kcol, lab in zip(kanton_cols, kanton_labels)
    }

heatmap_df = pd.DataFrame(rows).T
heatmap_df.index.name = "akteur"

# optional: nur Akteure mit mindestens einem gültigen Wert
heatmap_df = heatmap_df.dropna(how="all")

heatmap_df